In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/student-por.csv", sep=";")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (649, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [3]:
grade_bins = [0, 10, 14, 17, 21]

grade_labels = [
    "Poor",
    "Average",
    "Good",
    "Excellent"
]

df["Performance"] = pd.cut(
    df["G3"],
    bins=grade_bins,
    labels=grade_labels,
    right=False
)

print(df[["G3", "Performance"]].head(10))

   G3 Performance
0  11     Average
1  11     Average
2  12     Average
3  14        Good
4  13     Average
5  13     Average
6  13     Average
7  13     Average
8  17   Excellent
9  13     Average


In [4]:
print(df["Performance"].value_counts().sort_index())

Performance
Poor         100
Average      355
Good         148
Excellent     46
Name: count, dtype: int64


In [6]:
X = df.drop(columns=["G3", "Performance"])
y = df["Performance"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (649, 32)
Target shape: (649,)


In [7]:
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("\nNumber of categorical features:", len(categorical_features))

print("\nNumerical features:", numerical_features)
print("\nNumber of numerical features:", len(numerical_features))

Categorical features: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of categorical features: 17

Numerical features: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']

Number of numerical features: 15


C:\Users\Parv\AppData\Local\Temp\ipykernel_19564\2759884302.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [8]:
# Early prediction: no previous period grades
X_early = X.drop(columns=["G1", "G2"])

# Late prediction: previous period grades included
X_late = X.copy()

print("Early prediction features:", X_early.shape)
print("Late prediction features:", X_late.shape)

Early prediction features: (649, 30)
Late prediction features: (649, 32)


In [9]:
print("Class distribution:")
print(y.value_counts())

print("\nClass distribution (%):")
print(y.value_counts(normalize=True) * 100)

Class distribution:
Performance
Average      355
Good         148
Poor         100
Excellent     46
Name: count, dtype: int64

Class distribution (%):
Performance
Average      54.699538
Good         22.804314
Poor         15.408320
Excellent     7.087827
Name: proportion, dtype: float64


In [10]:
X_train_early, X_temp_early, y_train_early, y_temp_early = train_test_split(
    X_early,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val_early, X_test_early, y_val_early, y_test_early = train_test_split(
    X_temp_early,
    y_temp_early,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_early
)

In [11]:
X_train_late = X_late.loc[X_train_early.index]
X_val_late = X_late.loc[X_val_early.index]
X_test_late = X_late.loc[X_test_early.index]

y_train = y_train_early
y_val = y_val_early
y_test = y_test_early

In [12]:
print("EARLY PREDICTION")
print("Training:", X_train_early.shape)
print("Validation:", X_val_early.shape)
print("Testing:", X_test_early.shape)

print("\nLATE PREDICTION")
print("Training:", X_train_late.shape)
print("Validation:", X_val_late.shape)
print("Testing:", X_test_late.shape)

EARLY PREDICTION
Training: (454, 30)
Validation: (97, 30)
Testing: (98, 30)

LATE PREDICTION
Training: (454, 32)
Validation: (97, 32)
Testing: (98, 32)


In [13]:
for name, target in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name} distribution:")
    print(target.value_counts(normalize=True).sort_index().round(3))


Train distribution:
Performance
Poor         0.154
Average      0.546
Good         0.229
Excellent    0.070
Name: proportion, dtype: float64

Validation distribution:
Performance
Poor         0.155
Average      0.546
Good         0.227
Excellent    0.072
Name: proportion, dtype: float64

Test distribution:
Performance
Poor         0.153
Average      0.551
Good         0.224
Excellent    0.071
Name: proportion, dtype: float64


In [14]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [15]:
categorical_features = X_early.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_early.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

Categorical: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Numerical: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']


C:\Users\Parv\AppData\Local\Temp\ipykernel_19564\1681362276.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_early.select_dtypes(


In [16]:
def create_preprocessor(numerical_features, categorical_features):

    numerical_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    preprocessor = ColumnTransformer([
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return preprocessor

In [17]:
# Early prediction
preprocessor_early = create_preprocessor(
    numerical_features,
    categorical_features
)

# Late prediction
late_categorical_features = X_late.select_dtypes(
    include=["object", "category"]
).columns.tolist()

late_numerical_features = X_late.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor_late = create_preprocessor(
    late_numerical_features,
    late_categorical_features
)

C:\Users\Parv\AppData\Local\Temp\ipykernel_19564\4083767223.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  late_categorical_features = X_late.select_dtypes(


In [18]:
X_train_early_processed = preprocessor_early.fit_transform(
    X_train_early
)

X_val_early_processed = preprocessor_early.transform(
    X_val_early
)

X_test_early_processed = preprocessor_early.transform(
    X_test_early
)

In [19]:
X_train_late_processed = preprocessor_late.fit_transform(
    X_train_late
)

X_val_late_processed = preprocessor_late.transform(
    X_val_late
)

X_test_late_processed = preprocessor_late.transform(
    X_test_late
)

In [21]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Class mapping:")

for index, category in enumerate(label_encoder.classes_):
    print(category, ":", index)

Class mapping:
Average : 0
Excellent : 1
Good : 2
Poor : 3


In [23]:
from sklearn.feature_selection import mutual_info_classif

In [26]:
from sklearn.feature_selection import mutual_info_classif

# Get transformed feature names
feature_names = preprocessor_early.get_feature_names_out()

# Identify categorical (one-hot encoded) features
discrete_mask = np.array([
    name.startswith("categorical__")
    for name in feature_names
])

# Calculate Mutual Information
mi_scores = mutual_info_classif(
    X_train_early_processed,
    y_train_encoded,
    discrete_features=discrete_mask,
    random_state=42
)

# Create feature importance dataframe
mi_results = pd.DataFrame({
    "Feature": feature_names,
    "MI_Score": mi_scores
})

mi_results = mi_results.sort_values(
    by="MI_Score",
    ascending=False
)

mi_results.head(20)

,Feature,MI_Score
5,numerical__failures,0.083156
50,categorical__higher_no,0.054657
51,categorical__higher_yes,0.054657
14,categorical__school_MS,0.049211
13,categorical__school_GP,0.049211
8,numerical__goout,0.035417
0,numerical__age,0.032097
9,numerical__Dalc,0.028743
11,numerical__health,0.020172
12,numerical__absences,0.017956


In [27]:
np.savez(
    "../models/preprocessed_data.npz",

    X_train_early=X_train_early_processed,
    X_val_early=X_val_early_processed,

    y_train=y_train_encoded,
    y_val=y_val_encoded
)

In [28]:
np.savez_compressed(
    "../models/preprocessed_data.npz",

    X_train_early=X_train_early_processed,
    X_val_early=X_val_early_processed,
    X_test_early=X_test_early_processed,

    X_train_late=X_train_late_processed,
    X_val_late=X_val_late_processed,
    X_test_late=X_test_late_processed,

    y_train=y_train_encoded,
    y_val=y_val_encoded,
    y_test=y_test_encoded
)

print("All preprocessed data saved successfully!")

All preprocessed data saved successfully!
